# 01 — Reactive agent

**Definition:** look at the input, decide once, act, stop. No loop.

I'm starting here because every other pattern in this repo is literally "reactive **plus**
something" — plus a loop, plus a plan, plus a critic, plus memory. So this is the baseline
I keep comparing against.

```
Input  ->  Decide  ->  Act  ->  Output
              (no loop, no memory, no plan)
```

The whole thing is a function: `state -> action`. Stateless, no look-ahead, no look-back.

**The line that separates reactive from everything else:** the graph has **no cycle**.
Fan-out is fine, branching is fine, nothing may point backwards.

What I'm building: a customer-support triage graph.

```
        START
          |
          v
      classify          <- the ONLY LLM call
          |
          v   (conditional edge = routing, NOT looping)
   +------+-------+--------+
   v      v       v        v
billing  tech   refund  general
   +------+-------+--------+
          |
          v
         END
```

## Setup

The same cell opens every notebook here. Keys live in a `.env` at the repo root
(`GROQ_API_KEY=...`), which is gitignored.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


print(llm.invoke("Reply with the single word: ready").content)

In [ ]:
def show(graph):
    """Print a graph as mermaid text.

    graph.get_graph().draw_mermaid_png() renders a real image in Jupyter instead,
    but it calls out to mermaid.ink, so I keep the offline version as the default.
    """
    print(graph.get_graph().draw_mermaid())

## State

State is a `TypedDict` shared by every node. Each node returns a **partial** dict and
LangGraph merges it in. No reducer on a key means the returned value **overwrites** the
old one (last write wins) — that's exactly what I want here.

In [ ]:
from typing import Literal, TypedDict

from pydantic import BaseModel, Field


class ReactiveState(TypedDict):
    query: str      # input
    category: str   # written by classify
    response: str   # written by whichever handler runs


class Classification(BaseModel):
    """Pydantic schema -> the LLM has to emit valid, parseable output.

    Far more reliable than regexing free text, and `Literal` means the model
    physically cannot invent a fifth category.
    """

    category: Literal["billing", "technical", "refund", "general"] = Field(
        description="The support category this query belongs to"
    )

### Poking at `with_structured_output` first

Before wiring any graph, I want to see what the classifier alone returns.

**Gotcha worth knowing up front.** `with_structured_output` defaults to
`method="function_calling"` — it hands the model a fake tool and hopes it calls it. On
Groq's gpt-oss models that fails a lot for anything bigger than one field:

```
BadRequestError: Tool choice is required, but model did not call a tool
```

`method="json_schema"` constrains decoding to the schema instead of relying on the model
choosing to call a tool. It's the reliable option, so I use it everywhere in this repo.

In [ ]:
classifier = llm.with_structured_output(Classification, method="json_schema")

probe = classifier.invoke("Classify this customer support query.\n\nQuery: I was charged twice!")
print(type(probe), "->", probe)
print("just the field:", probe.category)

## Nodes

Every node has the same signature: `def node(state) -> dict` returning a partial update.

Note that only `classify` calls the LLM. The handlers are plain Python on purpose — a
reactive agent should push as much as possible into cheap, testable code.

In [ ]:
def classify(state: ReactiveState) -> dict:
    result = classifier.invoke(
        f"Classify this customer support query.\n\nQuery: {state['query']}"
    )
    print(f"  [classify] -> {result.category}")
    return {"category": result.category}   # return ONLY the key I changed


def handle_billing(state: ReactiveState) -> dict:
    return {"response": "Routed to Billing. Avg. response time: 4 hours."}


def handle_technical(state: ReactiveState) -> dict:
    return {"response": "Routed to Tech Support. Please share your error logs."}


def handle_refund(state: ReactiveState) -> dict:
    return {"response": "Refund request created. Processed within 5 business days."}


def handle_general(state: ReactiveState) -> dict:
    return {"response": "Thanks for reaching out! An agent will reply shortly."}

## The router

A router is **not** a node. The distinction I keep having to re-learn:

| | node | router (conditional edge fn) |
|---|---|---|
| returns | a state update (`dict`) | a node **name** (`str`) |
| mutates state | yes | no — it only reads |

Returning a dict from a router silently does nothing to state. That bug cost me an hour once.

In [ ]:
def route(state: ReactiveState) -> str:
    return {
        "billing": "handle_billing",
        "technical": "handle_technical",
        "refund": "handle_refund",
        "general": "handle_general",
    }[state["category"]]

## Build and compile

Current v1 API — the old tutorials use `set_entry_point()` / `set_finish_point()`, which
are gone. It's `add_edge(START, ...)` and `add_edge(..., END)` now.

The third argument to `add_conditional_edges` (the path map) is optional, but without it
LangGraph can't know where the router might jump, so the drawn graph is wrong.

In [ ]:
from langgraph.graph import END, START, StateGraph

HANDLERS = ["handle_billing", "handle_technical", "handle_refund", "handle_general"]

builder = StateGraph(ReactiveState)

builder.add_node("classify", classify)
for name, fn in [
    ("handle_billing", handle_billing),
    ("handle_technical", handle_technical),
    ("handle_refund", handle_refund),
    ("handle_general", handle_general),
]:
    builder.add_node(name, fn)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, HANDLERS)   # fan out

for h in HANDLERS:
    builder.add_edge(h, END)     # every branch terminates. No back-edges == reactive.

graph = builder.compile()
show(graph)

## Run it

In [ ]:
queries = [
    "I was charged twice this month!",
    "The app crashes when I open settings",
    "I want my money back for last month's plan",
    "What are your office hours?",
]

for q in queries:
    result = graph.invoke({"query": q})    # invoke = run to completion
    print(f"Q: {q}\n  -> [{result['category']}] {result['response']}\n")

### Streaming, to see each node fire

Three stream modes worth knowing:

- `"updates"` — only what each node **returned**. Best for debugging.
- `"values"` — the full state after each step.
- `"messages"` — LLM tokens as they're generated.

In [ ]:
print("--- updates ---")
for chunk in graph.stream({"query": "My invoice looks wrong"}, stream_mode="updates"):
    print(chunk)

print("\n--- values ---")
for chunk in graph.stream({"query": "My invoice looks wrong"}, stream_mode="values"):
    print(chunk)

## Experiment: what happens with no reducer?

I claimed above that a plain key overwrites. Quick check — two nodes both writing `note`.

In [ ]:
class OverwriteState(TypedDict):
    note: str


def first(state):
    return {"note": "written by first"}


def second(state):
    return {"note": "written by second"}


ob = StateGraph(OverwriteState)
ob.add_node("first", first)
ob.add_node("second", second)
ob.add_edge(START, "first")
ob.add_edge("first", "second")
ob.add_edge("second", END)

print(ob.compile().invoke({"note": "initial"}))
print("^ last write wins. Pattern 3 shows the `operator.add` reducer that appends instead.")

## Variant: reactive agent *with* a tool

This is the version people mistake for ReAct. It calls a tool — but exactly once, formats
the result, and stops. The tool result never goes back to the LLM, so there's no cycle.

In [ ]:
from langchain_core.tools import tool


@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order by its ID."""
    return {"A1": "Shipped, arrives Friday", "B2": "Processing"}.get(order_id, "Not found")


class ToolState(TypedDict):
    query: str
    response: str


def reactive_tool_node(state: ToolState) -> dict:
    """Decide -> act -> format. Exactly one pass."""
    model = llm.bind_tools([get_order_status])
    ai_msg = model.invoke(state["query"])

    if not ai_msg.tool_calls:
        return {"response": ai_msg.content}      # no tool needed, answer directly

    # Run the FIRST tool call, then stop.
    # A ReAct agent would instead feed this result back to the LLM. That's the whole diff.
    call = ai_msg.tool_calls[0]
    tool_result = get_order_status.invoke(call["args"])
    return {"response": f"Order {call['args']['order_id']}: {tool_result}"}


tb = StateGraph(ToolState)
tb.add_node("act", reactive_tool_node)
tb.add_edge(START, "act")
tb.add_edge("act", END)
tool_graph = tb.compile()

print(tool_graph.invoke({"query": "Where is order A1?"})["response"])
print(tool_graph.invoke({"query": "Tell me a joke about databases."})["response"][:120])

## Notes to self

**Reactive vs ReAct** — the one table I actually need:

| | reactive | ReAct |
|---|---|---|
| tool result goes | to the user | back into the LLM |
| graph shape | `START -> node -> END` (a DAG) | `START -> agent <-> tools -> END` (a cycle) |
| LLM calls | fixed, known upfront | unbounded until the LLM stops |

**Good fit:** classification, routing, triage, translation, extraction, "call one API and
format the answer". Anything where I already know the steps at design time.

**Bad fit:** the task needs discovery ("find X, then depending on X do Y"), several
dependent tool calls, or recovery from its own mistakes.

**Costs/benefits:** 1 LLM call, lowest latency, highest predictability. Nothing else here
is cheaper.

**API I used:**

```python
class S(TypedDict): ...                       # state
builder.add_node("name", fn)                  # node
builder.add_edge("a", "b")                    # fixed edge
builder.add_conditional_edges("a", router, ["b", "c"])   # branch
builder.add_edge(START, "a") / add_edge("a", END)
graph = builder.compile()
graph.invoke({...}) / graph.stream({...}, stream_mode="updates")
llm.with_structured_output(Model, method="json_schema")   # reliable parsing
```

Next: **02 — ReAct**, where I add the back-edge and the agent starts discovering its own path.